[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/area_transport/area_transport.ipynb)

# Area Transport through a Collapse

This example tracks the edges, oriented face areas, and volume of a cube as a shear-compression deformation flattens it into a horizontal sheet. As the height vanishes, the volume drops to zero while the horizontal faces keep their full area and the sheared side faces still cover part of the sheet. Carrying surface flux measurements back from the deformed faces to the undeformed cube ordinarily relies on an inverse deformation, which ceases to exist once the cube collapses flat.

A linear deformation is modelled as a map on point displacements, and its outermorphism `deformation.outermorphism(Patch)` moves each oriented surface patch by moving the two edges joined in `a & b` (`areas(a & b) == deformation(a) & deformation(b)`), while `deformation.det()` scales volumes. Pulling a flux density back through the area map uses its adjugate `areas.adjugate()`, defined directly by the flux pairing `flux_pullback(flux) & patch == flux & areas(patch)` with no inverse, so at zero volume it stays well defined and annihilates patches spanning the collapsed direction. In projective space, the adjugate `placement.adjugate()` of a combined deformation and translation transports planes with their position across `plane & placement(point)` even though the degenerate metric admits no adjoint, extending Nanson's area relation through singular collapse.

In continuum-mechanics matrix notation, a deformation gradient $F$ scales volumes by $J = \det F$ and maps unit-normal area elements $\mathbf{N}\,dA$ by [Nanson's area relation](https://my.mech.utah.edu/~brannon/public/Deformation.pdf) $\mathbf{n}\,da = (\det F)\,F^{-\mathsf{T}}\mathbf{N}\,dA$, while pulling a flux vector $\mathbf{q}$ back to the reference configuration writes $\mathbf{Q} = (\det F)\,F^{-1}\mathbf{q}$ and requires replacing $(\det F)\,F^{-\mathsf{T}}$ with the explicit cofactor matrix $\operatorname{cof} F$ and adjugate $(\operatorname{cof} F)^{\mathsf{T}}$ whenever $\det F = 0$. Including a translation $\mathbf{t}$ embeds points as homogeneous $4$-vectors $[x, y, z, 1]^{\mathsf{T}}$ under a block matrix $H = \bigl[\begin{smallmatrix} F & \mathbf{t} \\ \mathbf{0}^{\mathsf{T}} & 1 \end{smallmatrix}\bigr]$ and transforms plane coefficients $[n_x, n_y, n_z, -d]$ separately by $H^{-\mathsf{T}}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from itertools import product

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D, VGA3D
from examples.util.animation import show_animation
from examples.mechanics.area_transport import render

np.set_printoptions(precision=3, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
# Points, and edge displacements between them, are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()                               # Antivector-
# An oriented patch, the join of two edge displacements, carries its face's attitude, area, and sense.
Patch = ga.gatype.antibivector()                             # Antibivector-
# A flux density pairs with an oriented patch to give the flux through it.
Flux = ga.gatype.bivector()                                  # Bivector+
Linear = ga.gatype((Point, Point))                           # Point- <- Point-
AreaMap = ga.gatype((Patch, Patch))                          # Patch- <- Patch-
FluxMap = ga.gatype((Flux, Flux))                            # Flux+ <- Flux+

pga = NumpyContext(PGA3D).multivector
ProjectivePlane = PGA3D.gatype.vector()                      # Vector+
ProjectivePoint = PGA3D.gatype.antivector()                  # Antivector-
Placement = PGA3D.gatype((ProjectivePoint, ProjectivePoint)) # ProjectivePoint- <- ProjectivePoint-

## Area transport and flux pullback through a collapse

The identity `Point` maps every point displacement to itself, while pairing `z & Point` reads a point's height so vertical compression and horizontal shear add displacements proportional to that height. Joining two edge displacements `a & b` forms an intrinsic oriented patch with the attitude, area, and winding sense of its face, and `deformation.outermorphism(Patch)` moves patches by moving both spanning edges while `deformation.det()` scales volumes.

Pairing an extrinsic flux density with an intrinsic patch via `flux & patch` yields the scalar flux through the face. The adjugate `areas.adjugate()` pulls any flux density back across that pairing so `flux_pullback(flux) & patch == flux & areas(patch)` holds at every thickness, and at the flat sheet the pulled-back horizontal flux vanishes on every patch spanning the crushed direction.

In [ ]:
side = 1.0
n_frames = 36
duration_ms = 40
max_shear = 0.70
# Ease the height from the unit cube down to a flat sheet while shearing horizontal layers along x.
thicknesses = (1 + np.cos(np.linspace(0, np.pi, n_frames))) / 2
shears = max_shear * (1 - thicknesses)

In [ ]:
def cube(side: float) -> tuple[Point, Point, Patch]:
    """A centred cube of edge length `side` has eight corners (`[8] Point-`), six face centres (`[6] Point-`), and six outward-wound face patches (`[6] Patch-`)."""
    corners = mv.vector(list(product((-side / 2, side / 2), repeat=3))).dual()  # [8] Point-
    axes = stack((mv.x, mv.y, mv.z))                                            # [3] Plane+
    face_planes = concatenate([axes, -axes])                                    # [6] Plane+
    centres = face_planes.dual() * (side / 2)                                   # [6] Point-
    # Join pairs of edge displacements into face patches, then wind every face outward with the cube's volume.
    edges = axes.dual() * side                                                  # [3] Point-
    x_edge, y_edge, z_edge = edges[0], edges[1], edges[2]                       # [] Point-, Point-, Point-
    axis_patches = stack((y_edge & z_edge, z_edge & x_edge, x_edge & y_edge))   # [3] Patch-
    raw_patches = concatenate([axis_patches, axis_patches])                     # [6] Patch-
    volume_sense = x_edge & y_edge & z_edge                                     # [] Scalar-
    orientation = (raw_patches & centres) * volume_sense                        # [6] Scalar+
    patches = raw_patches * (orientation / orientation.norm())                  # [6] Patch-
    return corners, centres, patches


def deformation(thickness: np.ndarray, shear: np.ndarray) -> Linear:
    """Vertical compression by `thickness` and horizontal layer shear by `shear` map point displacements linearly (`[...] Point- <- Point-`)."""
    height = mv.z & Point                                                       # [] Scalar+ <- Point-
    return Point + (thickness - 1) * mv.z.dual() * height + shear * mv.x.dual() * height


rest_corners, rest_centres, rest_patches = cube(side)                           # [8] Point-, [6] Point-, [6] Patch-
crush = deformation(thicknesses, shears)                                        # [n_frames] Point- <- Point-

In [ ]:
def transport(deformation: Linear) -> tuple[AreaMap, FluxMap, Scalar]:
    """The deformation's outermorphism (`[...] Patch- <- Patch-`), its flux-pullback adjugate (`[...] Flux+ <- Flux+`), and its determinant (`[...] Scalar+`) govern area, flux, and volume transport."""
    # Move the two spanning edges of each oriented patch together.
    areas = deformation.outermorphism(Patch)                                    # [...] Patch- <- Patch-
    # Pull flux densities back across the area pairing without requiring an inverse.
    flux_pullback = areas.adjugate()                                            # [...] Flux+ <- Flux+
    return areas, flux_pullback, deformation.det()


areas, flux_pullback, volumes = transport(crush)                                # [n_frames] Patch- <- Patch-, Flux+ <- Flux+, Scalar+
vertices = crush[:, None](rest_corners)                                         # [n_frames, 8] Point-
centres = crush[:, None](rest_centres)                                          # [n_frames, 6] Point-
patches = areas[:, None](rest_patches)                                          # [n_frames, 6] Patch-
horizontal_flux = flux_pullback(mv.xy)                                          # [n_frames] Flux+
# Span a patch with the collapsed z-shear direction and y to test annihilation at the flat sheet.
null_patch = (mv.z - shears[-1] * mv.x).dual() & mv.y.dual()                    # [] Patch-
show_animation(render.animate(zip(vertices, centres, patches, volumes)), "area_transport", duration_ms,
               forward_then_back=True);

The animation shears and compresses the unit cube into a flat horizontal sheet and back, drawing each face as a translucent plate with an outward arrow whose length equals the face's area. As the height reaches zero, the horizontal face keeps unit area, the tilted side faces retain nonzero projected area on the sheet, the crushed vertical face loses its arrow, and the four bars track the three face areas and vanishing volume against the dotted unit-cube level.

## Plane transport in projective space

In projective space, combining vertical compression, horizontal shear, and translation along `x` produces a single linear map `ProjectivePoint <- ProjectivePoint` on positioned points. Its adjugate `placement.adjugate()` preserves plane-point incidence `plane & placement(point)` across the map, so solving `plane_pullback.solve(face_planes)` carries the cube's six bounding planes forward with both their tilted attitudes and translated offsets, whereas `placement.adjoint()` fails because the inner product on projective points sees only point weight and is singular.

In homogeneous matrix coordinates, ordering the plane basis `(x, y, z, w)` against its complementary point basis `(wzy, wxz, wyx, xyz)` makes the adjugate's coefficient matrix the transpose $M^{\mathsf{T}}$ of the $4 \times 4$ point matrix $M$.

In [ ]:
sample_thickness = 0.35
sample_shear = 0.45
translation = 0.5

In [ ]:
def projective_placement(thickness: float, shear: float, translation: float) -> Placement:
    """Vertical compression by `thickness`, shear by `shear`, and translation along `x` by `translation` combine into one projective point map (`[] ProjectivePoint- <- ProjectivePoint-`)."""
    height = pga.z & ProjectivePoint                                            # [] Scalar+ <- ProjectivePoint-
    squeeze = (ProjectivePoint
               + (thickness - 1) * pga.z.dual() * height
               + shear * pga.x.dual() * height)                                 # [] ProjectivePoint- <- ProjectivePoint-
    translator = (pga.xw * (translation / 2)).exp()                             # [] Rotor+
    return translator >> squeeze


def projective_cube(side: float) -> tuple[ProjectivePoint, ProjectivePlane]:
    """A centred projective cube of edge length `side` has eight corner points (`[8] ProjectivePoint-`) bounded by six face planes (`[6] ProjectivePlane+`)."""
    corner_points = pga.point(list(product((-side / 2, side / 2), repeat=3)))   # [8] ProjectivePoint-
    axes = stack((pga.x, pga.y, pga.z))                                         # [3] ProjectivePlane+
    face_planes = concatenate([axes, -axes]) - pga.w * (side / 2)               # [6] ProjectivePlane+
    return corner_points, face_planes


placement = projective_placement(sample_thickness, sample_shear, translation)   # [] ProjectivePoint- <- ProjectivePoint-
plane_pullback = placement.adjugate()                                           # [] ProjectivePlane+ <- ProjectivePlane+
corner_points, face_planes = projective_cube(side)                              # [8] ProjectivePoint-, [6] ProjectivePlane+
image_planes = plane_pullback.solve(face_planes)                                # [6] ProjectivePlane+

print(f"placement\n{placement.coefficient_table()}\n")
print(f"adjugate\n{plane_pullback.coefficient_table()}\n")
try:
    placement.adjoint()
except np.linalg.LinAlgError as error:
    print("adjoint:", error)

In [ ]:
# checks
# Pulling horizontal flux back gives the same flux through every undeformed face as through the deformed face.
assert (((horizontal_flux[:, None] & rest_patches[None, :]) - (mv.xy & patches)).abs() < 1e-12).all()
# The deformation and its area map scale every face pyramid's volume by the determinant, including at zero volume.
assert (((centres & patches) - volumes[:, None] * (rest_centres & rest_patches)).norm() < 1e-12).all()
# At the flat sheet, the pulled-back horizontal flux vanishes on the collapsed patch that the area map crushes.
assert (horizontal_flux[-1] & null_patch).abs() < 1e-12
assert areas[-1](null_patch).norm() < 1e-12
# Projective plane pullback preserves every face-corner incidence across shear, compression, and translation.
rest_incidence = face_planes[:, None] & corner_points[None, :]                  # [6, 8] Scalar+
image_incidence = image_planes[:, None] & placement(corner_points)[None, :]     # [6, 8] Scalar+
assert ((image_incidence - rest_incidence).abs() < 1e-12).all()